# News-to-Signal Case Study

## Research Question
Can simple text-derived features from financial news be transformed into
a cross-sectional ranking signal with measurable predictive behavior?

## Project Goal
This notebook demonstrates a reproducible quant research workflow:
from raw news text → structured features → signal construction → evaluation.
The goal is research process clarity, not production-ready performance.

> **About the outputs.** Saved outputs were cleared on purpose. The last saved run had only 44 ticker-days, eight
> dates with an IC and one date with a long-short spread: too few to support any conclusion, and a "hit rate" over
> one date means nothing. Run the notebook on your own data, with API keys in `.env`, to reproduce the workflow.


In [ ]:
# Setup: add parent directory to path for imports
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Import project modules
from src import preprocess, feature_engineering, signal_construction, evaluation
from config import FIGURES_DIR, TABLES_DIR

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print("Imports successful!")

## Section 2 — Data Assembly and Preprocessing

This section fetches financial news from Yahoo Finance RSS and price data
from yfinance. It computes forward returns and performs data quality checks.

**Pipeline steps:**
1. Fetch news headlines for all tickers in the universe
2. Fetch daily close prices with retry and cache logic
3. Compute future_return_5d for each headline
4. Aggregate headlines by ticker-date
5. Run data quality assertions

In [ ]:
# Run the preprocessing pipeline
# Note: This may take a few minutes as it fetches data from external APIs
df_raw = preprocess.main()
print(f"\nFinal dataset shape: {df_raw.shape}")

In [ ]:
# Display first few rows
df_raw.head()

In [ ]:
# Summary statistics
df_raw.describe()

In [ ]:
# Plot headline count distribution
fig, ax = plt.subplots(figsize=(10, 6))
df_raw['headline_count'].hist(bins=20, edgecolor='black', ax=ax)
ax.set_xlabel('Headlines per Day')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of Headlines per Ticker-Date')
ax.axvline(df_raw['headline_count'].mean(), color='red', linestyle='--',
           label=f"Mean: {df_raw['headline_count'].mean():.2f}")
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/headline_count_distribution.png', dpi=150)
plt.show()

## Section 3 — Feature Engineering

This section transforms raw news text into structured features using the
Loughran-McDonald (LMD) Master Dictionary.

**Features computed:**
- **sentiment_score**: (positive_count - negative_count) / total_words, clipped to [-1, 1]
- **uncertainty_score**: uncertainty_count / total_words
- **event_intensity**: fraction of event keywords present in headline

In [ ]:
# Apply feature engineering
df_features = feature_engineering.main(df_raw)
print(f"\nFeatures added. New shape: {df_features.shape}")

In [ ]:
# Plot histograms for the three features
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

features = ['sentiment_score', 'uncertainty_score', 'event_intensity']
titles = ['Sentiment Score', 'Uncertainty Score', 'Event Intensity']

for ax, feature, title in zip(axes, features, titles):
    df_features[feature].hist(bins=30, edgecolor='black', ax=ax)
    ax.set_xlabel(title)
    ax.set_ylabel('Frequency')
    ax.set_title(f'Distribution of {title}')
    ax.axvline(df_features[feature].mean(), color='red', linestyle='--',
               label=f"Mean: {df_features[feature].mean():.3f}")
    ax.legend()

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/feature_distributions.png', dpi=150)
plt.show()

## Section 4 — Signal Construction

This section applies cross-sectional z-score standardization and constructs
three distinct signals.

**Signals:**
- **signal_sentiment_only** = z(sentiment_score)
- **signal_sentiment_minus_uncertainty** = z(sentiment_score) - z(uncertainty_score)
- **signal_full** = z(sentiment_score) - z(uncertainty_score) + z(event_intensity)

All z-scores are computed independently within each trading date (cross-sectional).

In [ ]:
# Apply signal construction
df_signals = signal_construction.main(df_features)
print(f"\nSignals constructed. Final shape: {df_signals.shape}")

In [ ]:
# Plot distribution of the full signal
fig, ax = plt.subplots(figsize=(10, 6))
df_signals['signal_full'].hist(bins=40, edgecolor='black', ax=ax)
ax.set_xlabel('Signal Full (z-score standardized)')
ax.set_ylabel('Frequency')
ax.set_title('Distribution of signal_full')
ax.axvline(df_signals['signal_full'].mean(), color='red', linestyle='--',
           label=f"Mean: {df_signals['signal_full'].mean():.3f}")
ax.legend()
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/signal_full_distribution.png', dpi=150)
plt.show()

## Section 4.5 — Baseline Comparison

The baseline comparison tests whether adding uncertainty and event intensity
contributes incremental signal value beyond sentiment alone.

In [ ]:
# Run baseline comparison with statistical significance
summary = evaluation.run_baseline_comparison(df_signals)
print("Signal Performance Summary with Statistical Significance")
print("=" * 60)
display(summary)

# Highlight significance
if 'significant_5pct' in summary.columns:
    print("\nStatistically Significant Signals (p < 0.05):")
    significant = summary[summary['significant_5pct'] == True]
    if len(significant) > 0:
        for _, row in significant.iterrows():
            print(f"  • {row['signal']}: t-stat={row['ic_t_statistic']:.2f}, p={row['ic_p_value']:.4f}")
    else:
        print("  No signals significant at 5% level (expected with limited sample)")

In [ ]:
# Plot mean IC comparison
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(summary))
width = 0.35

bars1 = ax.bar(x - width/2, summary['mean_ic'], width, label='Mean IC', color='steelblue')
bars2 = ax.bar(x + width/2, summary['mean_spread'], width, label='Mean Spread', color='coral')

ax.set_xlabel('Signal')
ax.set_ylabel('Value')
ax.set_title('Signal Performance Comparison')
ax.set_xticks(x)
ax.set_xticklabels(summary['signal'], rotation=15, ha='right')
ax.legend()
ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.3f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom', fontsize=8)

for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.4f}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/baseline_comparison.png', dpi=150)
plt.show()

In [ ]:
# Plot hit rates
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(summary))
width = 0.35

bars1 = ax.bar(x - width/2, summary['ic_hit_rate'], width, label='IC Hit Rate', color='steelblue')
bars2 = ax.bar(x + width/2, summary['spread_hit_rate'], width, label='Spread Hit Rate', color='coral')

ax.set_xlabel('Signal')
ax.set_ylabel('Hit Rate')
ax.set_title('Signal Hit Rate Comparison (above 50% = better than random)')
ax.set_xticks(x)
ax.set_xticklabels(summary['signal'], rotation=15, ha='right')
ax.legend()
ax.axhline(y=0.5, color='red', linestyle='--', linewidth=1, label='Random (50%)')
ax.set_ylim(0, 1)

# Add value labels on bars
for bar in bars1:
    height = bar.get_height()
    ax.annotate(f'{height:.1%}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom', fontsize=8)

for bar in bars2:
    height = bar.get_height()
    ax.annotate(f'{height:.1%}',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}/hit_rate_comparison.png', dpi=150)
plt.show()

## Section 5 — Evaluation

All evaluation is cross-sectional (per trading date). We compute:
- **Grouped forward returns**: Mean return per quintile/tercile
- **Top-minus-Bottom spread**: Long top group, short bottom group
- **Spearman Rank IC**: Rank correlation between signal and forward return
- **Monthly stability**: Consistency of daily spreads over time

In [ ]:
# Compute grouped returns for signal_full
grouped_returns = evaluation.compute_grouped_returns(df_signals, 'signal_full')
print(f"Grouped returns computed for {grouped_returns['date'].nunique()} dates")

# Compute overall mean by group
mean_by_group = grouped_returns.groupby('group')['mean_return'].mean()
print("\nMean return by group:")
print(mean_by_group)

In [ ]:
# Plot grouped returns
if len(mean_by_group) > 0:
    fig, ax = plt.subplots(figsize=(10, 6))

    mean_by_group.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
    ax.set_xlabel('Group (1 = Lowest Signal, N = Highest Signal)')
    ax.set_ylabel('Mean Forward Return (5-day)')
    ax.set_title('Mean Forward Return by Signal Group (signal_full)')
    ax.axhline(y=0, color='red', linestyle='--', linewidth=1)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

    # Add value labels
    for i, v in enumerate(mean_by_group.values):
        ax.annotate(f'{v:.4f}',
                    xy=(i, v),
                    xytext=(0, 5 if v >= 0 else -15),
                    textcoords="offset points",
                    ha='center', va='bottom' if v >= 0 else 'top', fontsize=10)

    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/grouped_returns.png', dpi=150)
    plt.show()
else:
    print("WARNING: No grouped returns available. Insufficient data to form groups.")
    print("This happens when each date has fewer than MIN_SAMPLE_DROP (9) observations.")
    print("Consider expanding the ticker universe or the lookback period.")

In [ ]:
# Compute daily spread for signal_full
spread_series = evaluation.compute_daily_spread(df_signals, 'signal_full')
print(f"Daily spread computed for {len(spread_series)} dates")

if len(spread_series) > 0:
    print(f"Mean spread: {spread_series.mean():.4f}")
    print(f"Spread hit rate: {(spread_series > 0).mean():.1%}")
else:
    print("WARNING: No daily spreads available. Insufficient cross-sectional data per date.")

In [ ]:
# Plot daily spread over time
if len(spread_series) > 0:
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))

    # Time series
    ax1 = axes[0]
    spread_series.plot(ax=ax1, color='steelblue', alpha=0.7)
    ax1.axhline(y=spread_series.mean(), color='red', linestyle='--',
                label=f"Mean: {spread_series.mean():.4f}")
    ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Daily Spread')
    ax1.set_title('Daily Top-Minus-Bottom Spread (signal_full)')
    ax1.legend()

    # Histogram
    ax2 = axes[1]
    spread_series.hist(bins=20, edgecolor='black', ax=ax2, color='steelblue', alpha=0.7)
    ax2.axvline(spread_series.mean(), color='red', linestyle='--',
                label=f"Mean: {spread_series.mean():.4f}")
    ax2.axvline(0, color='black', linestyle='-', linewidth=0.5)
    ax2.set_xlabel('Daily Spread')
    ax2.set_ylabel('Frequency')
    ax2.set_title('Distribution of Daily Spreads')
    ax2.legend()

    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/daily_spread.png', dpi=150)
    plt.show()
else:
    print("Skipping daily spread plot - no data available.")

In [ ]:
# Compute daily IC
ic_series = evaluation.compute_daily_ic(df_signals, 'signal_full')
print(f"Daily IC computed for {len(ic_series)} dates")

if len(ic_series) > 0:
    print(f"Mean IC: {ic_series.mean():.4f}")
    print(f"IC hit rate: {(ic_series > 0).mean():.1%}")
    print(f"IC std: {ic_series.std():.4f}")
    print(f"IR (IC/Std): {ic_series.mean() / ic_series.std():.4f}")
else:
    print("WARNING: No daily ICs available. Insufficient cross-sectional data per date.")

In [ ]:
# Plot daily IC
if len(ic_series) > 0:
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))

    # Time series
    ax1 = axes[0]
    ic_series.plot(ax=ax1, color='coral', alpha=0.7)
    ax1.axhline(y=ic_series.mean(), color='red', linestyle='--',
                label=f"Mean IC: {ic_series.mean():.4f}")
    ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax1.set_xlabel('Date')
    ax1.set_ylabel('Daily IC')
    ax1.set_title('Daily Information Coefficient (signal_full)')
    ax1.legend()

    # Histogram
    ax2 = axes[1]
    ic_series.hist(bins=20, edgecolor='black', ax=ax2, color='coral', alpha=0.7)
    ax2.axvline(ic_series.mean(), color='red', linestyle='--',
                label=f"Mean: {ic_series.mean():.4f}")
    ax2.axvline(0, color='black', linestyle='-', linewidth=0.5)
    ax2.set_xlabel('Daily IC')
    ax2.set_ylabel('Frequency')
    ax2.set_title('Distribution of Daily ICs')
    ax2.legend()

    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/daily_ic.png', dpi=150)
    plt.show()
else:
    print("Skipping daily IC plot - no data available.")

In [ ]:
# Compute monthly stability
monthly_spread = evaluation.compute_monthly_stability(spread_series)
print("Monthly mean spreads:")
print(monthly_spread if len(monthly_spread) > 0 else "No data available.")

In [ ]:
# Plot monthly stability
if len(monthly_spread) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))

    monthly_spread.plot(ax=ax, marker='o', color='steelblue', linewidth=2, markersize=8)
    ax.axhline(y=monthly_spread.mean(), color='red', linestyle='--',
               label=f"Overall Mean: {monthly_spread.mean():.4f}")
    ax.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax.set_xlabel('Month')
    ax.set_ylabel('Mean Spread')
    ax.set_title('Monthly Stability of Daily Spreads')
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/monthly_stability.png', dpi=150)
    plt.show()
else:
    print("Skipping monthly stability plot - no data available.")

# Run walk-forward validation
from src.walkforward import run_walkforward_analysis, check_robustness

print("Running Walk-Forward Analysis...")
print("Split: Train (180d) / Validation (90d) / Test (95d)")
print("=" * 60)

wf_results = run_walkforward_analysis(df_signals)
display(wf_results)

# Check robustness
print("\nRobustness Assessment:")
print("=" * 60)
robustness = check_robustness(wf_results)
for signal, assessment in robustness.items():
    status = "✓ ROBUST" if assessment['robust'] else "✗ NOT ROBUST"
    print(f"\n{signal}: {status}")
    print(f"  Sign consistency: {assessment['sign_consistent']}")
    print(f"  Any significant: {assessment['any_significant']}")
    if assessment['deterioration'] is not None:
        print(f"  Val→Test deterioration: {assessment['deterioration']:.1%}")

# Plot walk-forward results
if len(wf_results) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Mean IC by period
    ax1 = axes[0]
    pivot_ic = wf_results.pivot(index='signal', columns='period', values='mean_ic')
    pivot_ic = pivot_ic[['train', 'validation', 'test']]  # Reorder columns
    pivot_ic.plot(kind='bar', ax=ax1, color=['steelblue', 'orange', 'coral'])
    ax1.set_title('Mean IC Across Walk-Forward Periods')
    ax1.set_ylabel('Mean Information Coefficient')
    ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax1.legend(title='Period')
    ax1.set_xticklabels(ax1.get_xticklabels(), rotation=15, ha='right')
    
    # Sharpe ratio by period
    ax2 = axes[1]
    pivot_sharpe = wf_results.pivot(index='signal', columns='period', values='sharpe_ratio')
    pivot_sharpe = pivot_sharpe[['train', 'validation', 'test']]
    pivot_sharpe.plot(kind='bar', ax=ax2, color=['steelblue', 'orange', 'coral'])
    ax2.set_title('Sharpe Ratio Across Walk-Forward Periods')
    ax2.set_ylabel('Sharpe Ratio (Annualized)')
    ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    ax2.legend(title='Period')
    ax2.set_xticklabels(ax2.get_xticklabels(), rotation=15, ha='right')
    
    plt.tight_layout()
    plt.savefig(f'{FIGURES_DIR}/walkforward_analysis.png', dpi=150)
    plt.show()
else:
    print("No walk-forward results to plot.")

In [ ]:
## Section 8 — Conclusion

### Key Findings

1. **Signal Efficacy**: The full signal (sentiment - uncertainty + event intensity) demonstrates [directional consistency / statistical significance] across evaluation periods.

2. **Statistical Robustness**: 
   - Mean IC: [value]
   - IC t-statistic: [value] ([significant / not significant] at 5% level)
   - Annualized Information Ratio: [value]

3. **Walk-Forward Validation**: Signal [maintained / deteriorated] out-of-sample performance relative to in-sample, suggesting [generalizability / overfitting concerns].

4. **Risk Profile**: Sharpe ratio [value], maximum drawdown [value], Sortino ratio [value].

### Research Limitations

1. **Sample Constraints**: 50-stock universe limits statistical power and cross-sectional diversity.
2. **Survivorship Bias**: Current large-cap selection introduces look-ahead bias.
3. **News Coverage**: Yahoo Finance RSS has incomplete coverage; premium sources would improve signal strength.
4. **Holding Period**: Fixed 5-day horizon ignores optimal exit timing.

### Methodological Strengths

- **Rigorous Evaluation**: IC significance testing, walk-forward validation, risk-adjusted metrics
- **Modular Design**: Reusable components for rapid iteration
- **Transparency**: Full documentation of assumptions and limitations

### Transferable Skills Demonstrated

| Skill | Evidence |
|-------|----------|
| Alpha Research | End-to-end signal construction pipeline |
| Statistical Testing | t-statistics, p-values, information ratios |
| Risk Management | Sharpe, Sortino, drawdown analysis |
| Time-Series Validation | Train/validation/test splits |
| Python Engineering | Modular, documented, type-hinted code |

### Next Steps for Production

1. **Neutralization**: Add sector/market cap neutralization to control factor exposures
2. **Transaction Costs**: Model market impact and slippage for realistic P&L
3. **Ensemble Methods**: Combine with alternative signals (momentum, value)
4. **Alternative Data**: Integrate social media, earnings call transcripts

### Research Note

This study prioritizes methodological rigor over absolute performance. The goal is to demonstrate quantitative research capabilities—including hypothesis formation, data engineering, statistical validation, and risk-aware evaluation—essential for quantitative researcher roles in systematic asset management.

In [ ]:
import os

# Ensure directories exist
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)

# Save summary tables
summary.to_csv(f'{TABLES_DIR}/baseline_comparison.csv', index=False)
print(f"Saved baseline comparison to {TABLES_DIR}/baseline_comparison.csv")

# Save walk-forward results
if 'wf_results' in globals() and len(wf_results) > 0:
    wf_results.to_csv(f'{TABLES_DIR}/walkforward_results.csv', index=False)
    print(f"Saved walk-forward results to {TABLES_DIR}/walkforward_results.csv")

# Save grouped returns (if available)
if 'mean_by_group' in globals() and len(mean_by_group) > 0:
    mean_by_group.to_csv(f'{TABLES_DIR}/mean_returns_by_group.csv')
    print(f"Saved grouped returns to {TABLES_DIR}/mean_returns_by_group.csv")
else:
    print("No grouped returns to save.")

# Save daily metrics (if available)
if 'spread_series' in globals() and len(spread_series) > 0:
    metrics_df = pd.DataFrame({
        'date': spread_series.index,
        'spread': spread_series.values,
    })
    metrics_df.to_csv(f'{TABLES_DIR}/daily_spread.csv', index=False)
    print(f"Saved daily spread to {TABLES_DIR}/daily_spread.csv")
else:
    print("No daily spread data to save.")

if 'ic_series' in globals() and len(ic_series) > 0:
    ic_df = pd.DataFrame({
        'date': ic_series.index,
        'ic': ic_series.values,
    })
    ic_df.to_csv(f'{TABLES_DIR}/daily_ic.csv', index=False)
    print(f"Saved daily IC to {TABLES_DIR}/daily_ic.csv")
else:
    print("No daily IC data to save.")

# Save monthly stability (if available)
if 'monthly_spread' in globals() and len(monthly_spread) > 0:
    monthly_spread.to_csv(f'{TABLES_DIR}/monthly_stability.csv')
    print(f"Saved monthly stability to {TABLES_DIR}/monthly_stability.csv")
else:
    print("No monthly stability data to save.")

print("\nAll results exported successfully!")